# Load an Existing Chroma Database

This notebook reconnects to the persisted Chroma collection created in the CRUD notebook and reads its contents.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

## 1. Rebuild the Same Configuration

In [2]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('c:/Users/Shivam Singh/advanced-rag-loaders/04_vector_stores')

In [3]:
# Load environment variables from the local .env file.

dotenv_path = project_root / ".env"
load_dotenv(dotenv_path=dotenv_path)

if not os.getenv("GOOGLE_API_KEY"):
    raise ValueError("Please add your GOOGLE_API_KEY to the .env file before running this notebook.")

print(f"Loaded environment from: {dotenv_path}")

Loaded environment from: c:\Users\Shivam Singh\advanced-rag-loaders\04_vector_stores\.env


In [4]:
collection_name = "demo"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo
Persist directory: c:\Users\Shivam Singh\advanced-rag-loaders\04_vector_stores\notebooks\chroma_langchain_db


In [5]:
# Create the embedding model and connect it to a persistent Chroma store.
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)

print("Connection established.")

Connection established.


## 2. Add Small Display Helpers

In [6]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_stored_documents(records):
    """Print stored Chroma records in a readable format."""
    ids = records.get("ids", [])
    documents = records.get("documents", [])
    metadatas = records.get("metadatas", [])

    print(f"Total documents in collection: {len(ids)}")
    print()

    for index, (doc_id, document_text, metadata) in enumerate(zip(ids, documents, metadatas), start=1):
        print(f"{index}. id={doc_id}")
        print(f"   topic={metadata.get('topic')} | doc_number={metadata.get('doc_number')}")
        print(f"   content={preview_text(document_text)}")
        print()

## 3. Fetch and Inspect the Stored Records

In [7]:
stored_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
stored_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [8]:
stored_records["embeddings"].shape

(8, 3072)

In [9]:
print_stored_documents(stored_records)

Total documents in collection: 8

1. id=e34a0406-678a-4c32-abdd-4f794bc02a3e
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human rea...

2. id=ff4a83c8-538a-4c92-8cd9-8982f3e366d4
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.

3. id=f8c3409d-4445-4b6a-bc7c-f2ce57a7229a
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.

4. id=7345df51-dc72-46aa-89d3-1ff14289105a
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language m...

5. id=da20cd2e-df63-4f87-8b95-9752cde17df6
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model ge...

6. id=8a9ed068-8a37-4b50-b8b4-63eb991a77ab
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they make semantic 